<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/riesgo/notebooks/c4_l8.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C4-L8 · Diario y kill-switch | 45 días, freno en −3,00%: 6 activaciones, drawdown máximo −8,77%.

In [ ]:
# CELDA COLAB-FIRST: correla primero si estas en Google Colab.
# Descarga el CSV del repo; si falla (sin red ), usa el CSV local.
import pandas as pd
from pathlib import Path

ORG = "Emelecto"  # organizacion fija del repo Emelecto/QuantLab
CSV_NOMBRE = "c4_l8.csv"
CSV_URL = f"https://raw.githubusercontent.com/{ORG}/QuantLab/main/web/content/cursos/riesgo/data/{CSV_NOMBRE}"

try:
    df = pd.read_csv(CSV_URL)
    print("CSV descargado desde:", CSV_URL)
except Exception as e:
    print("Uso CSV local (motivo:", str(e)[:80] + ")")
    csv_path = Path("../data") / CSV_NOMBRE
    if not csv_path.exists():
        csv_path = Path(CSV_NOMBRE)  # fallback si corres desde data/
    df = pd.read_csv(csv_path)
print(df.shape)
print(df.head())

In [ ]:
import numpy as np
df["equity"] = 100 * (1 + df["pnl_diario_pct"] / 100).cumprod()
df["peak"] = df["equity"].cummax()
df["dd_calc"] = (df["equity"] / df["peak"] - 1) * 100
kills = df[df["kill_switch"] == 1]
print("Activaciones:", len(kills), "en", len(df), "dias")
print("Drawdown maximo: %.2f%%" % df["dd_calc"].min())

## El freno abarata la vuelta | Con kill: −8,77% (volver = +9,6%). Sin freno: ~−21,7% (volver = casi +28%).

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df["dd_calc"].values, color="#f59e0b", label="drawdown (%)")
ax.scatter(kills.index, df.loc[kills.index, "dd_calc"], color="#f87171", zorder=5, label="kill-switch")
ax.set_xlabel("Dia")
ax.set_ylabel("Drawdown acumulado (%)")
ax.legend()
plt.show()

In [ ]:
print("Perdida evitada por el freno: 12,9 pp en los 6 dias de tilt.")
print("Kill mental no vale: alerta en -2,5%, cierre en -3,00%.")

In [ ]:
# Chequeos automáticos
assert len(df) == 45, "se esperan 45 dias"
assert df["kill_switch"].sum() == 6, "deben ser 6 activaciones"
assert ((df["pnl_diario_pct"] == -3.0) == (df["kill_switch"] == 1)).all(), "kill salta justo en -3,00%"
assert abs(df["dd_calc"].min() - (-8.77)) < 0.02, "drawdown maximo -8,77%"
print("OK: el freno funciona.")